# Ouroboros — real corpus from Hugging Face (Am1-A)

Downloads the real OCSR corpus from the Hugging Face Hub, detects its columns / splits / sources, writes one manifest per source, ingests each into shards on Drive (`MyDrive/ouroboros/real/<source>`) and runs the InChIKey leakage check against the synthetic training set. CPU-only. For a private dataset add a Colab secret `HF_TOKEN` (read access). **Check the inspect output before converting**: if a column, split or source was detected wrongly, set the overrides in the convert cell.

Colab's system Python (3.13) cannot install the pinned stack (escnn needs lie_learn and numpy<2, which have no 3.13 builds), so the setup creates a Python 3.11 virtualenv with `uv` at `/content/venv` and installs exactly `requirements.txt` (incl. torch 2.10.0 CUDA). All project code runs in that interpreter via subprocesses (`!{PY} ...`). If the repo is private, add a Colab secret `GITHUB_TOKEN` (key icon in the left sidebar) with read access to the repository.

In [ ]:
import time, os, subprocess, json
T0 = time.time()
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || true
!python --version && nproc && free -g | head -2
VENV_PY = '/content/venv/bin/python'  # project interpreter (created below)
# Colab's kernel exports MPLBACKEND=module://matplotlib_inline..., which does not exist in
# the venv; every project command therefore runs with an explicit headless backend.
PY = f'MPLBACKEND=Agg {VENV_PY}'

In [ ]:
# ---- configuration ----
REPO = 'yaniguan/ouroboros-ocsr'
BRANCH = 'claude/vigilant-johnson-j4882f'  # set to 'main' once merged
DRIVE_ROOT = '/content/drive/MyDrive/ouroboros'  # data, runs, results live here
REPO_DIR = '/content/ouroboros-ocsr'
LOCAL_DATA = '/content/data/full'  # configs expect shards at /content/data/full/shards
# the kernel's inline matplotlib backend does not exist in the project venv (subprocesses)
os.environ['MPLBACKEND'] = 'Agg'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
for sub in ('data', 'runs', 'results', 'real'):
    os.makedirs(f'{DRIVE_ROOT}/{sub}', exist_ok=True)

In [ ]:
token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass
url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '-b', BRANCH, url, REPO_DIR], check=True)
os.chdir(REPO_DIR)
!git log --oneline -1

In [ ]:
%%bash -s "$REPO_DIR"
set -e
cd "$1"
# py3nj (escnn dependency) builds from source and needs a Fortran compiler
which gfortran || (apt-get -qq update && apt-get -qq install -y gfortran > /dev/null)
pip install -q uv
# Python 3.11 venv with the exact tested pins (Colab's system Python is 3.13)
[ -x /content/venv/bin/python ] || uv venv -q --python 3.11 --python-preference only-managed /content/venv
uv pip install -q --python /content/venv/bin/python -r requirements.txt
uv pip install -q --python /content/venv/bin/python --no-deps -e .
# startup hook: any python of this venv replaces an inherited inline matplotlib backend
SP=$(/content/venv/bin/python -c 'import site; print(site.getsitepackages()[0])')
cp scripts/colab_mplbackend.pth "$SP/"
export MPLBACKEND=Agg
/content/venv/bin/python -c "import sys, torch; print(sys.version.split()[0], 'torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
HF_REPO = 'yaniguan/ocsr-dataset'
HF_LOCAL = '/content/hf/ocsr-dataset'  # raw snapshot (local disk, not Drive)
HF_OUT = '/content/hf_real'  # extracted images + manifests (local disk)
REAL = f'{DRIVE_ROOT}/real'  # ingested shards, one directory per source
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    print('no HF_TOKEN secret: fine for a public dataset')

In [ ]:
# download + inspect (no files written besides the snapshot) -- report this output back
t = time.time()
!{PY} scripts/hf_to_manifest.py inspect --repo {HF_REPO} --local {HF_LOCAL} | tee {REAL}/hf_inspect.json
!du -sh {HF_LOCAL}
print(f'download + inspect {(time.time() - t) / 60:.1f} min')

In [ ]:
# convert: set overrides only if the inspect output detected something wrongly, e.g.
# OVERRIDES = '--smiles-col gt_smiles --source-col dataset'; SPLIT_MAP = 'validation=test'
OVERRIDES = ''
SPLIT_MAP = ''  # e.g. 'val=test'; rows without a split become 'test' (never trained on)
SPLIT_ARG = f'--split-map {SPLIT_MAP}' if SPLIT_MAP else ''
!rm -rf {HF_OUT}
!{PY} scripts/hf_to_manifest.py convert --local {HF_LOCAL} --out {HF_OUT} {OVERRIDES} {SPLIT_ARG}
!cp {HF_OUT}/hf_manifest_stats.json {REAL}/

In [ ]:
# ingest every source into its own shard directory (standardized labels, failures logged)
import glob
SOURCES = sorted(os.path.basename(m)[:-4] for m in glob.glob(f'{HF_OUT}/manifests/*.csv'))
print(SOURCES)
t = time.time()
for src in SOURCES:
    subprocess.run(['rm', '-rf', f'{REAL}/{src}'], check=True)
    !{PY} scripts/ingest_real.py --manifest {HF_OUT}/manifests/{src}.csv --out {REAL}/{src}
print(f'ingest {(time.time() - t) / 60:.1f} min')

In [ ]:
# leakage: every real val/test set vs synthetic train (if generated) and real train sets.
# Writes the eval InChIKeys for data.exclude_keys (load-time exclusion from training).
def has(src, split):
    return bool(glob.glob(f'{REAL}/{src}/{split}-*.tar'))
EVAL = ' '.join(f'{s}_{sp}={REAL}/{s}:{sp}' for s in SOURCES for sp in ('val', 'test') if has(s, sp))
TRAIN = ' '.join(f'{s}_train={REAL}/{s}:train' for s in SOURCES if has(s, 'train'))
synth = f'{DRIVE_ROOT}/data/full/manifests/train.tsv'
if os.path.exists(synth):
    TRAIN += f' synth1m={synth}'
print('eval:', EVAL, '\ntrain:', TRAIN)
!{PY} scripts/check_leakage.py --eval {EVAL} --train {TRAIN} --dump-eval-keys {REAL}/eval_keys.txt | tee {REAL}/leakage.txt

In [ ]:
# ---- report back ----
!cat {REAL}/hf_manifest_stats.json | head -60
for src in SOURCES:
    print('==', src)
    !cat {REAL}/{src}/ingest_stats.json && head -3 {REAL}/{src}/ingest_failures.jsonl
!cat {REAL}/leakage.txt; wc -l {REAL}/eval_keys.txt; du -sh {REAL}
print(f'total notebook {(time.time() - T0) / 60:.1f} min')